# Chapter 4: Knowledge Distillation Experiments

**Thesis:** Knowledge Distillation Techniques for Optimising Large Language Models in Resource-Constrained Environments

- **Tasks:** SST-2 (classification) · SQuAD v1.1 (extractive QA)
- **Teacher:** `bert-large-uncased` (336M params)
- **Student:** `distilbert-base-uncased` (66M params)
- **Methods:** Baseline (B0) · Logit-KD (KD1) · Feature-KD (KD3) · Sequence-KD (KD2) · Q8 · Q4
- **Environment:** Google Colab with H100 GPU

In [ ]:
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
!pip install -q transformers==4.40.0 datasets evaluate accelerate scikit-learn

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/kd_thesis'
for d in ['checkpoints', 'results/tables', 'results/figures', 'cache']:
    Path(f'{PROJECT}/{d}').mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT}')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json, time, gc, copy, warnings
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR, SequentialLR, ConstantLR

from transformers import (
    BertTokenizerFast, BertForSequenceClassification, BertForQuestionAnswering,
    DistilBertTokenizerFast, DistilBertForSequenceClassification, DistilBertForQuestionAnswering,
    DistilBertModel, get_linear_schedule_with_warmup
)
from datasets import load_dataset
import evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Configuration & Phase Switches

In [ ]:
RUN_TEACHER_SST2  = True
RUN_B0_SST2       = True
RUN_KD1_SST2      = True
RUN_KD3_SST2      = True
RUN_ABLATION      = True
RUN_TEACHER_SQUAD = True
RUN_B0_SQUAD      = True
RUN_KD1_SQUAD     = True
RUN_KD2_SQUAD     = True
RUN_QUANTIZATION  = True
RUN_BENCHMARK     = True
RUN_FIGURES       = True

CONFIG = {
    'teacher_sst2_model' : 'bert-large-uncased',
    'teacher_squad_model': 'deepset/bert-large-uncased-whole-word-masking-squad2',
    'student_model'      : 'distilbert-base-uncased',

    'seeds'              : [42, 43, 44],
    'sst2_epochs'        : 3,
    'squad_epochs'       : 2,
    'lr'                 : 2e-5,
    'weight_decay'       : 0.01,
    'warmup_ratio'       : 0.1,
    'max_grad_norm'      : 1.0,
    'batch_size'         : 32,
    'sst2_max_len'       : 128,
    'squad_max_len'      : 384,
    'squad_doc_stride'   : 128,

    'kd1_temperature'    : 4.0,
    'kd1_alpha'          : 0.5,
    'kd3_lambda'         : 0.5,
    'kd2_alpha'          : 0.7,

    'ablation_temps'     : [2.0, 4.0],
    'ablation_alphas'    : [0.5, 0.7, 0.9],

    'benchmark_runs'     : 100,
    'benchmark_warmup'   : 10,
}

print('Configuration ready')
print(json.dumps(CONFIG, indent=2))

## 2. Utility Functions

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def model_size_mb(model):
    p = sum(p.nelement() * p.element_size() for p in model.parameters())
    b = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (p + b) / 1024**2


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def save_result(data, path):
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)


def load_result(path):
    with open(path) as f:
        return json.load(f)


def make_optimizer_scheduler(model, train_steps, lr, weight_decay, warmup_ratio):
    no_decay = ['bias', 'LayerNorm.weight']
    params = [
        {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': weight_decay},
        {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ]
    optimizer = AdamW(params, lr=lr)
    warmup_steps = int(warmup_ratio * train_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, train_steps)
    return optimizer, scheduler


print('Utilities defined')

## 3. Data Loading & Preprocessing

In [ ]:
raw_sst2  = load_dataset('glue', 'sst2')
raw_squad = load_dataset('squad')

print(f'SST-2  train={len(raw_sst2["train"]):,}  val={len(raw_sst2["validation"]):,}')
print(f'SQuAD  train={len(raw_squad["train"]):,}  val={len(raw_squad["validation"]):,}')

In [ ]:
bert_tok    = BertTokenizerFast.from_pretrained('bert-large-uncased')
distil_tok  = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')


def tokenise_sst2(examples, tokenizer, max_len):
    return tokenizer(
        examples['sentence'],
        truncation=True, padding='max_length', max_length=max_len
    )


def prepare_sst2(tokenizer, max_len):
    fn = lambda ex: tokenise_sst2(ex, tokenizer, max_len)
    ds = raw_sst2.map(fn, batched=True, remove_columns=['sentence', 'idx'])
    ds = ds.rename_column('label', 'labels')
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds


sst2_bert   = prepare_sst2(bert_tok,   CONFIG['sst2_max_len'])
sst2_distil = prepare_sst2(distil_tok, CONFIG['sst2_max_len'])

print('SST-2 datasets prepared')

In [ ]:
def tokenise_squad_train(examples, tokenizer, max_len, doc_stride):
    tokenized = tokenizer(
        examples['question'], examples['context'],
        truncation='only_second', max_length=max_len,
        stride=doc_stride, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding='max_length'
    )

    sample_map  = tokenized.pop('overflow_to_sample_mapping')
    offset_map  = tokenized.pop('offset_mapping')

    start_positions, end_positions = [], []

    for i, offsets in enumerate(offset_map):
        sample_idx = sample_map[i]
        answer     = examples['answers'][sample_idx]
        cls_idx    = tokenized['input_ids'][i].index(tokenizer.cls_token_id)

        if len(answer['answer_start']) == 0:
            start_positions.append(cls_idx)
            end_positions.append(cls_idx)
            continue

        start_char = answer['answer_start'][0]
        end_char   = start_char + len(answer['text'][0])

        seq_ids = tokenized.sequence_ids(i)
        context_start = seq_ids.index(1)
        context_end   = len(seq_ids) - 1 - seq_ids[::-1].index(1)

        if not (offsets[context_start][0] <= start_char and offsets[context_end][1] >= end_char):
            start_positions.append(cls_idx)
            end_positions.append(cls_idx)
            continue

        tok_start = context_start
        while tok_start <= context_end and offsets[tok_start][0] <= start_char:
            tok_start += 1
        start_positions.append(tok_start - 1)

        tok_end = context_end
        while tok_end >= context_start and offsets[tok_end][1] >= end_char:
            tok_end -= 1
        end_positions.append(tok_end + 1)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions']   = end_positions
    return tokenized


def prepare_squad_train(tokenizer, max_len, doc_stride):
    fn  = lambda ex: tokenise_squad_train(ex, tokenizer, max_len, doc_stride)
    ds  = raw_squad['train'].map(
        fn, batched=True, remove_columns=raw_squad['train'].column_names
    )
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'start_positions', 'end_positions'])
    return ds


squad_bert_train   = prepare_squad_train(bert_tok,   CONFIG['squad_max_len'], CONFIG['squad_doc_stride'])
squad_distil_train = prepare_squad_train(distil_tok, CONFIG['squad_max_len'], CONFIG['squad_doc_stride'])

print(f'SQuAD train BERT={len(squad_bert_train):,}  DistilBERT={len(squad_distil_train):,}')

In [ ]:
def tokenise_squad_val(examples, tokenizer, max_len, doc_stride):
    tokenized = tokenizer(
        examples['question'], examples['context'],
        truncation='only_second', max_length=max_len,
        stride=doc_stride, return_overflowing_tokens=True,
        return_offsets_mapping=True, padding='max_length'
    )
    sample_map = tokenized.pop('overflow_to_sample_mapping')
    tokenized['example_id'] = [examples['id'][i] for i in sample_map]
    return tokenized


def prepare_squad_val(tokenizer, max_len, doc_stride):
    fn = lambda ex: tokenise_squad_val(ex, tokenizer, max_len, doc_stride)
    ds = raw_squad['validation'].map(
        fn, batched=True, remove_columns=raw_squad['validation'].column_names
    )
    return ds


squad_bert_val   = prepare_squad_val(bert_tok,   CONFIG['squad_max_len'], CONFIG['squad_doc_stride'])
squad_distil_val = prepare_squad_val(distil_tok, CONFIG['squad_max_len'], CONFIG['squad_doc_stride'])

squad_metric = evaluate.load('squad')
print('SQuAD validation datasets ready')

## 4. KD Loss Implementations

In [ ]:
def kd1_classification_loss(student_logits, teacher_logits, labels, alpha, T):
    ce   = F.cross_entropy(student_logits, labels)
    kl   = F.kl_div(
        F.log_softmax(student_logits / T, dim=-1),
        F.softmax(teacher_logits / T, dim=-1),
        reduction='batchmean'
    ) * (T ** 2)
    return alpha * ce + (1.0 - alpha) * kl


def kd1_qa_loss(s_start, s_end, t_start, t_end, start_pos, end_pos, alpha, T):
    ce_start = F.cross_entropy(s_start, start_pos)
    ce_end   = F.cross_entropy(s_end,   end_pos)
    ce       = (ce_start + ce_end) / 2

    kl_start = F.kl_div(
        F.log_softmax(s_start / T, dim=-1),
        F.softmax(t_start / T, dim=-1),
        reduction='batchmean'
    ) * (T ** 2)
    kl_end   = F.kl_div(
        F.log_softmax(s_end / T, dim=-1),
        F.softmax(t_end / T, dim=-1),
        reduction='batchmean'
    ) * (T ** 2)
    kl = (kl_start + kl_end) / 2

    return alpha * ce + (1.0 - alpha) * kl


class DistilBertWithProjection(nn.Module):
    """
    Wraps DistilBertForSequenceClassification and adds a linear projection
    to align student hidden states with BERT-large (1024-dim).
    """
    def __init__(self, base_model, teacher_hidden=1024):
        super().__init__()
        self.model      = base_model
        self.projection = nn.Linear(768, teacher_hidden, bias=False)

    def forward(self, input_ids, attention_mask, labels=None):
        out  = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True
        )
        cls_hidden  = out.hidden_states[-1][:, 0, :]
        projected   = self.projection(cls_hidden)
        return out, projected


def kd3_classification_loss(student_logits, teacher_hidden, student_projected, labels, lam):
    ce       = F.cross_entropy(student_logits, labels)
    feat_mse = F.mse_loss(student_projected, teacher_hidden.detach())
    return ce + lam * feat_mse


print('KD loss functions defined')

## 5. Training Loops

In [ ]:
def evaluate_sst2(model, loader, device):
    metric = evaluate.load('glue', 'sst2')
    model.eval()
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            out    = model(**batch)
            preds  = out.logits.argmax(-1)
            metric.add_batch(predictions=preds.cpu(), references=labels.cpu())
    return metric.compute()['accuracy']


def train_baseline_sst2(model, train_ds, val_ds, config, device, seed=42):
    set_seed(seed)
    train_loader = DataLoader(train_ds['train'],      batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds['validation'],   batch_size=config['batch_size'] * 2)

    total_steps  = len(train_loader) * config['sst2_epochs']
    optimizer, scheduler = make_optimizer_scheduler(
        model, total_steps, config['lr'], config['weight_decay'], config['warmup_ratio']
    )

    best_acc, best_state = 0.0, None
    model.to(device)

    for epoch in range(config['sst2_epochs']):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{config["sst2_epochs"]}'):
            batch = {k: v.to(device) for k, v in batch.items()}
            out   = model(**batch)
            loss  = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        acc = evaluate_sst2(model, val_loader, device)
        print(f'  Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}  val_acc={acc:.4f}')

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    return model, best_acc


def train_kd1_sst2(student, teacher, train_ds, val_ds, config, device, seed=42):
    set_seed(seed)
    train_loader = DataLoader(train_ds['train'],    batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds['validation'], batch_size=config['batch_size'] * 2)

    total_steps  = len(train_loader) * config['sst2_epochs']
    optimizer, scheduler = make_optimizer_scheduler(
        student, total_steps, config['lr'], config['weight_decay'], config['warmup_ratio']
    )

    T, alpha = config['kd1_temperature'], config['kd1_alpha']

    teacher.eval()
    teacher.to(device)
    student.to(device)

    best_acc, best_state = 0.0, None

    for epoch in range(config['sst2_epochs']):
        student.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f'KD1 Epoch {epoch+1}'):
            batch   = {k: v.to(device) for k, v in batch.items()}
            labels  = batch['labels']

            with torch.no_grad():
                t_inp = {'input_ids': batch['input_ids'], 'attention_mask': batch['attention_mask']}
                t_out = teacher(**t_inp)

            s_inp = {'input_ids': batch['input_ids'], 'attention_mask': batch['attention_mask']}
            s_out = student(**s_inp)

            loss  = kd1_classification_loss(s_out.logits, t_out.logits, labels, alpha, T)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        acc = evaluate_sst2(student, val_loader, device)
        print(f'  Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}  val_acc={acc:.4f}')

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(student.state_dict())

    student.load_state_dict(best_state)
    return student, best_acc


def train_kd3_sst2(student_proj, teacher, train_ds, val_ds, config, device, seed=42):
    set_seed(seed)
    train_loader = DataLoader(train_ds['train'],    batch_size=config['batch_size'], shuffle=True)
    val_loader   = DataLoader(val_ds['validation'], batch_size=config['batch_size'] * 2)

    total_steps  = len(train_loader) * config['sst2_epochs']
    optimizer, scheduler = make_optimizer_scheduler(
        student_proj, total_steps, config['lr'], config['weight_decay'], config['warmup_ratio']
    )

    lam = config['kd3_lambda']
    teacher.eval()
    teacher.to(device)
    student_proj.to(device)

    best_acc, best_state = 0.0, None

    for epoch in range(config['sst2_epochs']):
        student_proj.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f'KD3 Epoch {epoch+1}'):
            batch  = {k: v.to(device) for k, v in batch.items()}
            labels = batch['labels']
            inp    = {'input_ids': batch['input_ids'], 'attention_mask': batch['attention_mask']}

            with torch.no_grad():
                t_out = teacher(**inp, output_hidden_states=True)
                teacher_cls = t_out.hidden_states[-1][:, 0, :]

            s_out, student_proj_feat = student_proj(**inp, labels=labels)

            loss = kd3_classification_loss(s_out.logits, teacher_cls, student_proj_feat, labels, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student_proj.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        acc = evaluate_sst2(student_proj.model, val_loader, device)
        print(f'  Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}  val_acc={acc:.4f}')

        if acc > best_acc:
            best_acc   = acc
            best_state = copy.deepcopy(student_proj.state_dict())

    student_proj.load_state_dict(best_state)
    return student_proj, best_acc


print('SST-2 training functions defined')

In [ ]:
def extract_squad_answers(model, val_dataset, raw_val, tokenizer, device, batch_size=32):
    model.eval()
    val_ds = val_dataset.remove_columns(['example_id', 'offset_mapping'])
    val_ds.set_format('torch', columns=['input_ids', 'attention_mask'])

    all_start, all_end = [], []
    loader = DataLoader(val_ds, batch_size=batch_size)
    with torch.no_grad():
        for batch in tqdm(loader, desc='SQuAD inference'):
            batch = {k: v.to(device) for k, v in batch.items()}
            out   = model(**batch)
            all_start.extend(out.start_logits.cpu().numpy())
            all_end.extend(out.end_logits.cpu().numpy())

    example_id_to_features = {}
    for i, eid in enumerate(val_dataset['example_id']):
        example_id_to_features.setdefault(eid, []).append(i)

    predictions = []
    for example in raw_val:
        eid      = example['id']
        feat_idx = example_id_to_features.get(eid, [])
        if not feat_idx:
            predictions.append({'id': eid, 'prediction_text': ''})
            continue

        best_score, best_text = -1e9, ''
        for fi in feat_idx:
            offsets = val_dataset['offset_mapping'][fi]
            s_logits = all_start[fi]
            e_logits = all_end[fi]
            seq_ids  = [None if o == [0,0] else 1 for o in offsets]

            for s_idx in np.argsort(s_logits)[-10:][::-1]:
                for e_idx in np.argsort(e_logits)[-10:][::-1]:
                    if seq_ids[s_idx] is None or seq_ids[e_idx] is None:
                        continue
                    if e_idx < s_idx or (e_idx - s_idx + 1) > 30:
                        continue
                    score = s_logits[s_idx] + e_logits[e_idx]
                    if score > best_score:
                        best_score = score
                        start_char = offsets[s_idx][0]
                        end_char   = offsets[e_idx][1]
                        best_text  = example['context'][start_char:end_char]

        predictions.append({'id': eid, 'prediction_text': best_text})

    references = [{'id': ex['id'], 'answers': ex['answers']} for ex in raw_val]
    results    = squad_metric.compute(predictions=predictions, references=references)
    return results['exact_match'], results['f1']


def train_baseline_squad(model, train_ds, config, device, seed=42):
    set_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=config['batch_size'], shuffle=True)
    total_steps  = len(train_loader) * config['squad_epochs']
    optimizer, scheduler = make_optimizer_scheduler(
        model, total_steps, config['lr'], config['weight_decay'], config['warmup_ratio']
    )

    model.to(device)
    for epoch in range(config['squad_epochs']):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f'SQuAD B0 Epoch {epoch+1}'):
            batch = {k: v.to(device) for k, v in batch.items()}
            out   = model(**batch)
            loss  = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()
        print(f'  Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}')

    return model


def train_kd1_squad(student, teacher, train_ds_s, train_ds_t, config, device, seed=42):
    set_seed(seed)
    loader_s = DataLoader(train_ds_s, batch_size=config['batch_size'], shuffle=False)
    loader_t = DataLoader(train_ds_t, batch_size=config['batch_size'], shuffle=False)

    total_steps = len(loader_s) * config['squad_epochs']
    optimizer, scheduler = make_optimizer_scheduler(
        student, total_steps, config['lr'], config['weight_decay'], config['warmup_ratio']
    )

    T, alpha = config['kd1_temperature'], config['kd2_alpha']
    teacher.eval()
    teacher.to(device)
    student.to(device)

    for epoch in range(config['squad_epochs']):
        student.train()
        total_loss = 0.0
        for batch_s, batch_t in tqdm(zip(loader_s, loader_t), desc=f'KD1-SQuAD Epoch {epoch+1}',
                                     total=len(loader_s)):
            batch_s = {k: v.to(device) for k, v in batch_s.items()}
            batch_t = {k: v.to(device) for k, v in batch_t.items()}

            start_pos = batch_s['start_positions']
            end_pos   = batch_s['end_positions']

            with torch.no_grad():
                t_inp = {'input_ids': batch_t['input_ids'], 'attention_mask': batch_t['attention_mask']}
                t_out = teacher(**t_inp)

            s_inp = {'input_ids': batch_s['input_ids'], 'attention_mask': batch_s['attention_mask']}
            s_out = student(**s_inp)

            loss = kd1_qa_loss(
                s_out.start_logits, s_out.end_logits,
                t_out.start_logits, t_out.end_logits,
                start_pos, end_pos, alpha, T
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        print(f'  Epoch {epoch+1}: loss={total_loss/len(loader_s):.4f}')

    return student


def train_kd2_squad(student, teacher, train_ds_s, train_ds_t, config, device, seed=42):
    """
    Sequence-level KD: student trained on teacher-predicted span positions as pseudo-labels.
    """
    set_seed(seed)
    loader_s = DataLoader(train_ds_s, batch_size=config['batch_size'], shuffle=False)
    loader_t = DataLoader(train_ds_t, batch_size=config['batch_size'], shuffle=False)

    total_steps = len(loader_s) * config['squad_epochs']
    optimizer, scheduler = make_optimizer_scheduler(
        student, total_steps, config['lr'], config['weight_decay'], config['warmup_ratio']
    )

    alpha = config['kd2_alpha']
    teacher.eval()
    teacher.to(device)
    student.to(device)

    for epoch in range(config['squad_epochs']):
        student.train()
        total_loss = 0.0
        for batch_s, batch_t in tqdm(zip(loader_s, loader_t), desc=f'KD2-SQuAD Epoch {epoch+1}',
                                     total=len(loader_s)):
            batch_s = {k: v.to(device) for k, v in batch_s.items()}
            batch_t = {k: v.to(device) for k, v in batch_t.items()}

            hard_start = batch_s['start_positions']
            hard_end   = batch_s['end_positions']

            with torch.no_grad():
                t_inp        = {'input_ids': batch_t['input_ids'], 'attention_mask': batch_t['attention_mask']}
                t_out        = teacher(**t_inp)
                pseudo_start = t_out.start_logits.argmax(-1)
                pseudo_end   = t_out.end_logits.argmax(-1)

            seq_len  = batch_s['input_ids'].shape[1]
            pseudo_start = pseudo_start.clamp(0, seq_len - 1)
            pseudo_end   = torch.maximum(pseudo_end.clamp(0, seq_len - 1), pseudo_start)

            s_inp = {'input_ids': batch_s['input_ids'], 'attention_mask': batch_s['attention_mask']}
            s_out = student(**s_inp)

            ce_hard   = (F.cross_entropy(s_out.start_logits, hard_start)
                       + F.cross_entropy(s_out.end_logits,   hard_end)) / 2
            ce_pseudo = (F.cross_entropy(s_out.start_logits, pseudo_start)
                       + F.cross_entropy(s_out.end_logits,   pseudo_end)) / 2

            loss = alpha * ce_hard + (1.0 - alpha) * ce_pseudo
            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), config['max_grad_norm'])
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += loss.item()

        print(f'  Epoch {epoch+1}: loss={total_loss/len(loader_s):.4f}')

    return student


print('SQuAD training functions defined')

## 6. SST-2 Experiments

In [ ]:
sst2_results = {}

if RUN_TEACHER_SST2:
    print('=== Teacher (BERT-large) Fine-tuning on SST-2 ===')
    teacher_sst2 = BertForSequenceClassification.from_pretrained(
        CONFIG['teacher_sst2_model'], num_labels=2
    )
    teacher_sst2, teacher_acc = train_baseline_sst2(
        teacher_sst2, sst2_bert, sst2_bert, CONFIG, DEVICE, seed=42
    )
    sst2_results['teacher'] = {'accuracy': teacher_acc}
    torch.save(teacher_sst2.state_dict(), f'{PROJECT}/checkpoints/teacher_sst2.pt')
    print(f'Teacher SST-2 accuracy: {teacher_acc:.4f}')
else:
    teacher_sst2 = BertForSequenceClassification.from_pretrained(
        CONFIG['teacher_sst2_model'], num_labels=2
    )
    teacher_sst2.load_state_dict(torch.load(f'{PROJECT}/checkpoints/teacher_sst2.pt'))
    teacher_sst2.to(DEVICE)

In [ ]:
if RUN_B0_SST2:
    print('=== Baseline B0 (DistilBERT, no KD) on SST-2 ===')
    b0_accs = []
    for seed in CONFIG['seeds']:
        model = DistilBertForSequenceClassification.from_pretrained(
            CONFIG['student_model'], num_labels=2
        )
        model, acc = train_baseline_sst2(model, sst2_distil, sst2_distil, CONFIG, DEVICE, seed=seed)
        b0_accs.append(acc)
        torch.save(model.state_dict(), f'{PROJECT}/checkpoints/b0_sst2_seed{seed}.pt')
        print(f'  Seed {seed}: {acc:.4f}')

    sst2_results['b0'] = {
        'accuracy_mean': np.mean(b0_accs),
        'accuracy_std' : np.std(b0_accs),
        'runs'         : b0_accs
    }
    print(f'B0 SST-2: {np.mean(b0_accs):.4f} ± {np.std(b0_accs):.4f}')

In [ ]:
if RUN_KD1_SST2:
    print('=== KD1 Logit-Based Distillation on SST-2 ===')
    kd1_accs = []
    for seed in CONFIG['seeds']:
        student = DistilBertForSequenceClassification.from_pretrained(
            CONFIG['student_model'], num_labels=2
        )
        student, acc = train_kd1_sst2(student, teacher_sst2, sst2_distil, sst2_distil, CONFIG, DEVICE, seed=seed)
        kd1_accs.append(acc)
        torch.save(student.state_dict(), f'{PROJECT}/checkpoints/kd1_sst2_seed{seed}.pt')
        print(f'  Seed {seed}: {acc:.4f}')

    sst2_results['kd1'] = {
        'accuracy_mean': np.mean(kd1_accs),
        'accuracy_std' : np.std(kd1_accs),
        'runs'         : kd1_accs
    }
    print(f'KD1 SST-2: {np.mean(kd1_accs):.4f} ± {np.std(kd1_accs):.4f}')

In [ ]:
if RUN_KD3_SST2:
    print('=== KD3 Feature-Based Distillation on SST-2 ===')
    kd3_accs = []
    for seed in CONFIG['seeds']:
        base_student = DistilBertForSequenceClassification.from_pretrained(
            CONFIG['student_model'], num_labels=2
        )
        student_proj = DistilBertWithProjection(base_student, teacher_hidden=1024)
        student_proj, acc = train_kd3_sst2(
            student_proj, teacher_sst2, sst2_distil, sst2_distil, CONFIG, DEVICE, seed=seed
        )
        kd3_accs.append(acc)
        torch.save(student_proj.model.state_dict(), f'{PROJECT}/checkpoints/kd3_sst2_seed{seed}.pt')
        print(f'  Seed {seed}: {acc:.4f}')

    sst2_results['kd3'] = {
        'accuracy_mean': np.mean(kd3_accs),
        'accuracy_std' : np.std(kd3_accs),
        'runs'         : kd3_accs
    }
    print(f'KD3 SST-2: {np.mean(kd3_accs):.4f} ± {np.std(kd3_accs):.4f}')

In [ ]:
if RUN_ABLATION:
    print('=== Ablation Study: Temperature × Alpha (KD1, SST-2, seed=42) ===')
    ablation_rows = []

    for T in CONFIG['ablation_temps']:
        for alpha in CONFIG['ablation_alphas']:
            cfg = {**CONFIG, 'kd1_temperature': T, 'kd1_alpha': alpha}
            student = DistilBertForSequenceClassification.from_pretrained(
                CONFIG['student_model'], num_labels=2
            )
            student, acc = train_kd1_sst2(
                student, teacher_sst2, sst2_distil, sst2_distil, cfg, DEVICE, seed=42
            )
            ablation_rows.append({'Temperature': T, 'Alpha': alpha, 'Val_Accuracy': round(acc, 4)})
            print(f'  T={T}  alpha={alpha}  acc={acc:.4f}')

    ablation_df = pd.DataFrame(ablation_rows)
    ablation_df.to_csv(f'{PROJECT}/results/tables/table_ablation.csv', index=False)
    print(ablation_df.to_string(index=False))

In [ ]:
save_result(sst2_results, f'{PROJECT}/results/raw/sst2_results.json')
print('SST-2 results saved')
for key, val in sst2_results.items():
    if 'accuracy_mean' in val:
        print(f'  {key:8s}  acc={val["accuracy_mean"]:.4f} ± {val["accuracy_std"]:.4f}')
    else:
        print(f'  teacher   acc={val["accuracy"]:.4f}')

## 7. SQuAD Experiments

In [ ]:
squad_results = {}

if RUN_TEACHER_SQUAD:
    print('=== Loading Pre-trained BERT-large Teacher for SQuAD ===')
    teacher_squad = BertForQuestionAnswering.from_pretrained(CONFIG['teacher_squad_model'])
    teacher_squad.to(DEVICE)
    teacher_squad.eval()

    em, f1 = extract_squad_answers(
        teacher_squad, squad_bert_val, raw_squad['validation'], bert_tok, DEVICE
    )
    squad_results['teacher'] = {'em': em, 'f1': f1}
    print(f'Teacher SQuAD: EM={em:.2f}  F1={f1:.2f}')

In [ ]:
if RUN_B0_SQUAD:
    print('=== Baseline B0 (DistilBERT, no KD) on SQuAD ===')
    b0_ems, b0_f1s = [], []
    for seed in CONFIG['seeds']:
        model = DistilBertForQuestionAnswering.from_pretrained(CONFIG['student_model'])
        model = train_baseline_squad(model, squad_distil_train, CONFIG, DEVICE, seed=seed)
        torch.save(model.state_dict(), f'{PROJECT}/checkpoints/b0_squad_seed{seed}.pt')

        em, f1 = extract_squad_answers(
            model, squad_distil_val, raw_squad['validation'], distil_tok, DEVICE
        )
        b0_ems.append(em)
        b0_f1s.append(f1)
        print(f'  Seed {seed}: EM={em:.2f}  F1={f1:.2f}')

    squad_results['b0'] = {
        'em_mean': np.mean(b0_ems),  'em_std': np.std(b0_ems),
        'f1_mean': np.mean(b0_f1s),  'f1_std': np.std(b0_f1s),
    }
    print(f'B0 SQuAD: EM={np.mean(b0_ems):.2f} ± {np.std(b0_ems):.2f}  F1={np.mean(b0_f1s):.2f} ± {np.std(b0_f1s):.2f}')

In [ ]:
if RUN_KD1_SQUAD:
    print('=== KD1 Logit-Based Distillation on SQuAD ===')
    kd1_ems, kd1_f1s = [], []
    for seed in CONFIG['seeds']:
        student = DistilBertForQuestionAnswering.from_pretrained(CONFIG['student_model'])
        student = train_kd1_squad(
            student, teacher_squad,
            squad_distil_train, squad_bert_train,
            CONFIG, DEVICE, seed=seed
        )
        torch.save(student.state_dict(), f'{PROJECT}/checkpoints/kd1_squad_seed{seed}.pt')

        em, f1 = extract_squad_answers(
            student, squad_distil_val, raw_squad['validation'], distil_tok, DEVICE
        )
        kd1_ems.append(em)
        kd1_f1s.append(f1)
        print(f'  Seed {seed}: EM={em:.2f}  F1={f1:.2f}')

    squad_results['kd1'] = {
        'em_mean': np.mean(kd1_ems), 'em_std': np.std(kd1_ems),
        'f1_mean': np.mean(kd1_f1s), 'f1_std': np.std(kd1_f1s),
    }
    print(f'KD1 SQuAD: EM={np.mean(kd1_ems):.2f} ± {np.std(kd1_ems):.2f}  F1={np.mean(kd1_f1s):.2f} ± {np.std(kd1_f1s):.2f}')

In [ ]:
if RUN_KD2_SQUAD:
    print('=== KD2 Sequence-Level Distillation on SQuAD ===')
    kd2_ems, kd2_f1s = [], []
    for seed in CONFIG['seeds']:
        student = DistilBertForQuestionAnswering.from_pretrained(CONFIG['student_model'])
        student = train_kd2_squad(
            student, teacher_squad,
            squad_distil_train, squad_bert_train,
            CONFIG, DEVICE, seed=seed
        )
        torch.save(student.state_dict(), f'{PROJECT}/checkpoints/kd2_squad_seed{seed}.pt')

        em, f1 = extract_squad_answers(
            student, squad_distil_val, raw_squad['validation'], distil_tok, DEVICE
        )
        kd2_ems.append(em)
        kd2_f1s.append(f1)
        print(f'  Seed {seed}: EM={em:.2f}  F1={f1:.2f}')

    squad_results['kd2'] = {
        'em_mean': np.mean(kd2_ems), 'em_std': np.std(kd2_ems),
        'f1_mean': np.mean(kd2_f1s), 'f1_std': np.std(kd2_f1s),
    }
    print(f'KD2 SQuAD: EM={np.mean(kd2_ems):.2f} ± {np.std(kd2_ems):.2f}  F1={np.mean(kd2_f1s):.2f} ± {np.std(kd2_f1s):.2f}')

In [ ]:
save_result(squad_results, f'{PROJECT}/results/raw/squad_results.json')
print('SQuAD results saved')

## 8. Quantization

In [ ]:
def quantize_q8(model):
    q_model = copy.deepcopy(model).cpu()
    q_model = torch.quantization.quantize_dynamic(
        q_model, {nn.Linear}, dtype=torch.qint8
    )
    return q_model


def quantize_q4(model):
    q_model = copy.deepcopy(model).cpu()
    with torch.no_grad():
        for module in q_model.modules():
            if isinstance(module, nn.Linear):
                w        = module.weight.data.float()
                scale    = w.abs().max() / 7.0
                if scale > 0:
                    w_q  = (w / scale).round().clamp(-8, 7) * scale
                    module.weight.data = w_q.to(module.weight.dtype)
    return q_model


if RUN_QUANTIZATION:
    print('=== Quantization (Q8 and Q4) ===')

    best_sst2_model = DistilBertForSequenceClassification.from_pretrained(
        CONFIG['student_model'], num_labels=2
    )
    best_sst2_model.load_state_dict(
        torch.load(f'{PROJECT}/checkpoints/kd1_sst2_seed42.pt')
    )

    val_loader = DataLoader(sst2_distil['validation'], batch_size=64)

    q8_sst2_accs, q4_sst2_accs = [], []
    for seed in CONFIG['seeds']:
        set_seed(seed)
        q8_model = quantize_q8(best_sst2_model)
        q4_model = quantize_q4(best_sst2_model)

        q8_acc   = evaluate_sst2(q8_model, val_loader, torch.device('cpu'))
        q4_acc   = evaluate_sst2(q4_model, val_loader, torch.device('cpu'))

        q8_sst2_accs.append(q8_acc)
        q4_sst2_accs.append(q4_acc)
        print(f'  Seed {seed}: Q8={q8_acc:.4f}  Q4={q4_acc:.4f}')

    sst2_results['q8'] = {
        'accuracy_mean': np.mean(q8_sst2_accs),
        'accuracy_std' : np.std(q8_sst2_accs)
    }
    sst2_results['q4'] = {
        'accuracy_mean': np.mean(q4_sst2_accs),
        'accuracy_std' : np.std(q4_sst2_accs)
    }

    torch.save(q8_model.state_dict(), f'{PROJECT}/checkpoints/q8_sst2.pt')
    torch.save(q4_model.state_dict(), f'{PROJECT}/checkpoints/q4_sst2.pt')

    best_squad_student = DistilBertForQuestionAnswering.from_pretrained(CONFIG['student_model'])
    best_squad_student.load_state_dict(
        torch.load(f'{PROJECT}/checkpoints/kd1_squad_seed42.pt')
    )

    q8_squad_ems, q8_squad_f1s = [], []
    q4_squad_ems, q4_squad_f1s = [], []

    for seed in CONFIG['seeds']:
        set_seed(seed)
        q8_sq = quantize_q8(best_squad_student)
        q4_sq = quantize_q4(best_squad_student)

        em8, f18 = extract_squad_answers(
            q8_sq, squad_distil_val, raw_squad['validation'], distil_tok, torch.device('cpu')
        )
        em4, f14 = extract_squad_answers(
            q4_sq, squad_distil_val, raw_squad['validation'], distil_tok, torch.device('cpu')
        )
        q8_squad_ems.append(em8);  q8_squad_f1s.append(f18)
        q4_squad_ems.append(em4);  q4_squad_f1s.append(f14)
        print(f'  Seed {seed}: Q8 EM={em8:.2f} F1={f18:.2f}  |  Q4 EM={em4:.2f} F1={f14:.2f}')

    squad_results['q8'] = {
        'em_mean': np.mean(q8_squad_ems), 'em_std': np.std(q8_squad_ems),
        'f1_mean': np.mean(q8_squad_f1s), 'f1_std': np.std(q8_squad_f1s),
    }
    squad_results['q4'] = {
        'em_mean': np.mean(q4_squad_ems), 'em_std': np.std(q4_squad_ems),
        'f1_mean': np.mean(q4_squad_f1s), 'f1_std': np.std(q4_squad_f1s),
    }

    save_result(sst2_results,  f'{PROJECT}/results/raw/sst2_results.json')
    save_result(squad_results, f'{PROJECT}/results/raw/squad_results.json')
    print('Quantization complete and results saved')

## 9. Efficiency Benchmarking

In [ ]:
def benchmark_model(model, tokenizer, device, n_runs=100, warmup=10, seq_len=128, batch_size=1):
    model.eval()
    model.to(device)

    dummy_text = 'The quick brown fox jumps over the lazy dog ' * 8
    inputs     = tokenizer(
        dummy_text, return_tensors='pt',
        truncation=True, max_length=seq_len, padding='max_length'
    )
    inputs = {k: v.repeat(batch_size, 1).to(device) for k, v in inputs.items()
              if k in ['input_ids', 'attention_mask']}

    if device.type == 'cuda':
        starter = torch.cuda.Event(enable_timing=True)
        ender   = torch.cuda.Event(enable_timing=True)

    with torch.no_grad():
        for _ in range(warmup):
            model(**inputs)

    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            if device.type == 'cuda':
                starter.record()
                model(**inputs)
                ender.record()
                torch.cuda.synchronize()
                latencies.append(starter.elapsed_time(ender))
            else:
                t0 = time.perf_counter()
                model(**inputs)
                latencies.append((time.perf_counter() - t0) * 1000)

    peak_mem_mb = 0
    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            model(**inputs)
        peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2

    avg_lat   = np.mean(latencies)
    throughput = (batch_size * seq_len * 1000) / avg_lat

    return {
        'latency_ms'    : round(avg_lat, 2),
        'latency_std_ms': round(np.std(latencies), 2),
        'throughput_tps': round(throughput),
        'peak_mem_mb'   : round(peak_mem_mb),
        'model_size_mb' : round(model_size_mb(model)),
        'num_params_M'  : round(count_params(model) / 1e6, 1),
    }


print('Benchmarking function defined')

In [ ]:
if RUN_BENCHMARK:
    print('=== Efficiency Benchmarking ===')
    bench_results = {}

    n_runs  = CONFIG['benchmark_runs']
    warmup  = CONFIG['benchmark_warmup']

    model_configs = [
        ('Teacher (BERT-large)',  teacher_sst2,   bert_tok,   DEVICE),
    ]

    best_kd1 = DistilBertForSequenceClassification.from_pretrained(CONFIG['student_model'], num_labels=2)
    best_kd1.load_state_dict(torch.load(f'{PROJECT}/checkpoints/kd1_sst2_seed42.pt'))

    best_b0  = DistilBertForSequenceClassification.from_pretrained(CONFIG['student_model'], num_labels=2)
    best_b0.load_state_dict(torch.load(f'{PROJECT}/checkpoints/b0_sst2_seed42.pt'))

    model_configs += [
        ('Student B0',   best_b0,        distil_tok, DEVICE),
        ('Student KD1',  best_kd1,       distil_tok, DEVICE),
        ('Student Q8',   quantize_q8(best_kd1), distil_tok, torch.device('cpu')),
        ('Student Q4',   quantize_q4(best_kd1), distil_tok, torch.device('cpu')),
    ]

    for name, model, tok, dev in model_configs:
        print(f'  Benchmarking {name}...')
        res_gpu = benchmark_model(model, tok, dev, n_runs=n_runs, warmup=warmup)

        if dev.type == 'cuda':
            res_cpu = benchmark_model(model, tok, torch.device('cpu'), n_runs=20, warmup=5)
        else:
            res_cpu = res_gpu

        bench_results[name] = {
            'gpu_latency_ms'    : res_gpu['latency_ms'],
            'gpu_throughput_tps': res_gpu['throughput_tps'],
            'cpu_latency_ms'    : res_cpu['latency_ms'],
            'cpu_throughput_tps': res_cpu['throughput_tps'],
            'peak_mem_mb'       : res_gpu['peak_mem_mb'],
            'model_size_mb'     : res_gpu['model_size_mb'],
            'num_params_M'      : res_gpu['num_params_M'],
        }
        print(f'    GPU: {res_gpu["latency_ms"]:.1f} ms  {res_gpu["throughput_tps"]:,} tps')
        print(f'    CPU: {res_cpu["latency_ms"]:.1f} ms  {res_cpu["throughput_tps"]:,} tps')
        print(f'    Mem: {res_gpu["peak_mem_mb"]} MB  Size: {res_gpu["model_size_mb"]} MB')

    save_result(bench_results, f'{PROJECT}/results/raw/bench_results.json')
    print('Benchmarking complete')

## 10. Results Tables

In [ ]:
sst2_results  = load_result(f'{PROJECT}/results/raw/sst2_results.json')
squad_results = load_result(f'{PROJECT}/results/raw/squad_results.json')
bench_results = load_result(f'{PROJECT}/results/raw/bench_results.json')

label_map = {
    'teacher': 'Teacher (BERT-large)',
    'b0' : 'Student B0 (Baseline)',
    'kd1': 'Student KD1 (Logit-KD)',
    'kd2': 'Student KD2 (Sequence-KD)',
    'kd3': 'Student KD3 (Feature-KD)',
    'q8' : 'Student Q8 (INT8)',
    'q4' : 'Student Q4 (INT4)',
}

rows = []
for key in ['teacher', 'b0', 'kd1', 'kd3', 'q8', 'q4']:
    r   = sst2_results.get(key, {})
    b   = bench_results.get(label_map[key], {})

    if key == 'teacher':
        acc_str = f"{r['accuracy']:.4f}"
        std_str = 'N/A'
    else:
        acc_str = f"{r['accuracy_mean']:.4f}"
        std_str = f"{r['accuracy_std']:.4f}"

    rows.append({
        'Model'               : label_map[key],
        'Params (M)'          : b.get('num_params_M', ''),
        'SST-2 Acc (mean)'    : acc_str,
        'SST-2 Acc (std)'     : std_str,
        'GPU Latency (ms)'    : b.get('gpu_latency_ms', ''),
        'GPU Throughput (tps)': b.get('gpu_throughput_tps', ''),
        'CPU Latency (ms)'    : b.get('cpu_latency_ms', ''),
        'Peak Memory (MB)'    : b.get('peak_mem_mb', ''),
        'Model Size (MB)'     : b.get('model_size_mb', ''),
    })

sst2_table = pd.DataFrame(rows)
sst2_table.to_csv(f'{PROJECT}/results/tables/table_sst2_main.csv', index=False)

print('=== Table 4.1a: SST-2 Classification Results ===')
print(sst2_table.to_string(index=False))

In [ ]:
rows_sq = []
for key in ['teacher', 'b0', 'kd1', 'kd2', 'q8', 'q4']:
    r = squad_results.get(key, {})

    if key == 'teacher':
        em_str = f"{r['em']:.2f}"
        f1_str = f"{r['f1']:.2f}"
        em_std, f1_std = 'N/A', 'N/A'
    else:
        em_str = f"{r['em_mean']:.2f}"
        f1_str = f"{r['f1_mean']:.2f}"
        em_std = f"{r['em_std']:.2f}"
        f1_std = f"{r['f1_std']:.2f}"

    rows_sq.append({
        'Model'        : label_map[key],
        'EM (mean)'    : em_str,
        'EM (std)'     : em_std,
        'F1 (mean)'    : f1_str,
        'F1 (std)'     : f1_std,
    })

squad_table = pd.DataFrame(rows_sq)
squad_table.to_csv(f'{PROJECT}/results/tables/table_squad_main.csv', index=False)

print('=== Table 4.1b: SQuAD QA Results ===')
print(squad_table.to_string(index=False))

## 11. Figures

In [ ]:
if RUN_FIGURES:
    COLORS = {'teacher': '#2196F3', 'b0': '#9E9E9E', 'kd1': '#FF5722',
              'kd2': '#FF9800', 'kd3': '#4CAF50', 'q8': '#9C27B0', 'q4': '#E91E63'}
    FIG_PATH = f'{PROJECT}/results/figures'

    PARAMS = {
        'teacher': 336, 'b0': 66, 'kd1': 66, 'kd2': 66, 'kd3': 66, 'q8': 66, 'q4': 66
    }

    SST2_ACC = {
        k: (sst2_results[k].get('accuracy_mean') or sst2_results[k].get('accuracy'))
        for k in sst2_results
    }

    fig, ax = plt.subplots(figsize=(8, 5))
    for key in SST2_ACC:
        ax.scatter(PARAMS[key], SST2_ACC[key] * 100,
                   color=COLORS[key], s=120, zorder=5, label=label_map[key])
        ax.annotate(label_map[key].split('(')[0].strip(),
                    (PARAMS[key], SST2_ACC[key] * 100),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)
    ax.set_xlabel('Model Parameters (M)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title('Figure 1: Model Quality vs Size (SST-2)')
    ax.legend(loc='lower right', fontsize=7)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/fig1_quality_vs_size.png', dpi=150)
    plt.show()
    print('Figure 1 saved')

In [ ]:
if RUN_FIGURES:
    bench_keys   = ['Teacher (BERT-large)', 'Student B0', 'Student KD1', 'Student Q8', 'Student Q4']
    gpu_lats     = [bench_results[k]['gpu_latency_ms']  for k in bench_keys]
    model_sizes  = [bench_results[k]['model_size_mb']   for k in bench_keys]
    colors_list  = ['#2196F3', '#9E9E9E', '#FF5722', '#9C27B0', '#E91E63']

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, k in enumerate(bench_keys):
        ax.scatter(model_sizes[i], gpu_lats[i], color=colors_list[i], s=120, zorder=5)
        ax.annotate(k, (model_sizes[i], gpu_lats[i]),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)
    ax.set_xlabel('Model Size (MB)')
    ax.set_ylabel('GPU Latency (ms)')
    ax.set_title('Figure 2: Inference Latency vs Model Size')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/fig2_latency_vs_size.png', dpi=150)
    plt.show()
    print('Figure 2 saved')

In [ ]:
if RUN_FIGURES:
    throughputs  = [bench_results[k]['gpu_throughput_tps'] for k in bench_keys]
    x_pos        = np.arange(len(bench_keys))

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(x_pos, throughputs, color=colors_list, alpha=0.85, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, throughputs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 200,
                f'{val:,}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([k.replace(' (', '\n(') for k in bench_keys], fontsize=9)
    ax.set_ylabel('Throughput (tokens/sec)')
    ax.set_title('Figure 3: Inference Throughput Comparison')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/fig3_throughput.png', dpi=150)
    plt.show()
    print('Figure 3 saved')

In [ ]:
if RUN_FIGURES:
    mem_vals  = [bench_results[k]['peak_mem_mb']   for k in bench_keys]
    size_vals = [bench_results[k]['model_size_mb'] for k in bench_keys]

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(bench_keys))
    w = 0.35
    b1 = ax.bar(x - w/2, size_vals, w, label='Model Size', color='#90CAF9', edgecolor='white')
    b2 = ax.bar(x + w/2, mem_vals,  w, label='Peak Memory', color='#EF9A9A', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels([k.replace(' (', '\n(') for k in bench_keys], fontsize=9)
    ax.set_ylabel('Memory (MB)')
    ax.set_title('Figure 4: Memory Usage')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/fig4_memory.png', dpi=150)
    plt.show()
    print('Figure 4 saved')

In [ ]:
if RUN_FIGURES:
    pareto_keys  = ['teacher', 'b0', 'kd1', 'kd3', 'q8', 'q4']
    pareto_names = [label_map[k] for k in pareto_keys]
    pareto_acc   = [SST2_ACC[k] * 100 for k in pareto_keys]
    pareto_lat   = [
        bench_results[label_map[k]]['gpu_latency_ms']
        for k in pareto_keys
    ]
    pareto_colors = [COLORS[k] for k in pareto_keys]

    fig, ax = plt.subplots(figsize=(8, 5))
    for i, k in enumerate(pareto_keys):
        ax.scatter(pareto_lat[i], pareto_acc[i],
                   color=pareto_colors[i], s=150, zorder=5, label=pareto_names[i])
        ax.annotate(pareto_names[i].split('(')[0].strip(),
                    (pareto_lat[i], pareto_acc[i]),
                    textcoords='offset points', xytext=(6, 3), fontsize=8)

    pareto_sorted = sorted(zip(pareto_lat, pareto_acc))
    front_x, front_y = [pareto_sorted[0][0]], [pareto_sorted[0][1]]
    best_acc = pareto_sorted[0][1]
    for lat, acc in pareto_sorted[1:]:
        if acc >= best_acc:
            front_x.append(lat)
            front_y.append(acc)
            best_acc = acc
    ax.step(front_x, front_y, '--', color='#666', alpha=0.5, where='post', label='Pareto front')

    ax.set_xlabel('GPU Latency (ms)')
    ax.set_ylabel('SST-2 Accuracy (%)')
    ax.set_title('Figure 5: Pareto Frontier – Quality vs Efficiency')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/fig5_pareto.png', dpi=150)
    plt.show()
    print('Figure 5 saved')

In [ ]:
if RUN_FIGURES:
    kd_keys     = ['kd1', 'kd3']
    kd_gains    = [
        (sst2_results[k]['accuracy_mean'] - sst2_results['b0']['accuracy_mean']) * 100
        for k in kd_keys
    ]
    kd_labels   = ['KD1 (Logit)', 'KD3 (Feature)']
    kd_colors   = [COLORS['kd1'], COLORS['kd3']]

    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(kd_labels, kd_gains, color=kd_colors, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, kd_gains):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                f'+{val:.2f}%', ha='center', va='bottom', fontsize=10)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('Accuracy Gain over B0 (%)')
    ax.set_title('Figure 6: Distillation Gain vs KD Method')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{FIG_PATH}/fig6_kd_gain.png', dpi=150)
    plt.show()
    print('Figure 6 saved')

## 12. Statistical Significance (Table 4.3)

In [ ]:
from scipy import stats

b0_runs  = sst2_results['b0']['runs']
kd1_runs = sst2_results['kd1']['runs']
kd3_runs = sst2_results['kd3']['runs']

stat_rows = []
for label, runs in [('KD1 vs B0', kd1_runs), ('KD3 vs B0', kd3_runs)]:
    diff       = np.mean(runs) - np.mean(b0_runs)
    t_stat, p  = stats.ttest_ind(runs, b0_runs)
    pooled_std = np.sqrt((np.std(runs)**2 + np.std(b0_runs)**2) / 2)
    cohens_d   = diff / pooled_std if pooled_std > 0 else 0
    stat_rows.append({
        'Comparison'       : label,
        'Mean Difference'  : f'{diff*100:+.2f}%',
        'p-value'          : f'{p:.4f}',
        "Cohen's d"        : f'{cohens_d:.3f}',
        'Significant (p<0.05)': 'Yes' if p < 0.05 else 'No'
    })

stat_df = pd.DataFrame(stat_rows)
stat_df.to_csv(f'{PROJECT}/results/tables/table_statistics.csv', index=False)

print('=== Table 4.3: Statistical Significance ===')
print(stat_df.to_string(index=False))

## 13. Summary Report

In [ ]:
print('=' * 60)
print('CHAPTER 4 EXPERIMENT SUMMARY')
print('=' * 60)

print('\nSST-2 Classification:')
for k in ['teacher', 'b0', 'kd1', 'kd3', 'q8', 'q4']:
    r = sst2_results[k]
    acc = r.get('accuracy_mean') or r.get('accuracy')
    std = r.get('accuracy_std', 0.0)
    print(f'  {label_map[k]:35s}  Acc={acc*100:.2f}% ± {std*100:.2f}%')

print('\nSQuAD Question Answering:')
for k in ['teacher', 'b0', 'kd1', 'kd2', 'q8', 'q4']:
    r = squad_results[k]
    em = r.get('em_mean') or r.get('em')
    f1 = r.get('f1_mean') or r.get('f1')
    print(f'  {label_map[k]:35s}  EM={em:.2f}  F1={f1:.2f}')

print('\nEfficiency (GPU):')
for k in ['Teacher (BERT-large)', 'Student B0', 'Student KD1', 'Student Q8', 'Student Q4']:
    b = bench_results[k]
    print(f'  {k:30s}  Lat={b["gpu_latency_ms"]:5.1f}ms  Thr={b["gpu_throughput_tps"]:7,} tps  Mem={b["peak_mem_mb"]:5}MB')

print('\nAll tables and figures saved to:', f'{PROJECT}/results/')